# 07 · The `drug_exposure` table

Drugs the patient is prescribed, is given or takes. The predefined `drug_exposure`
definition follows the CDM and [THEMIS](https://ohdsi.github.io/Themis/drug_exposure.html)
conventions.

Each drug gets a required `kind` label, which decides the end date and the visit:

| Label | Means | Visit / provider |
|---|---|---|
| `administered` | given to the patient ("ceftriaxone given in the ED") | from the note |
| `prescribed` | started or changed in this note ("start amoxicillin") | from the note |
| `taking` | on the medication list | left empty (not started in this visit) |

Details: [the drug_exposure table](https://precisiondata.github.io/aatm/getting-started/omop_tables/#the-drug_exposure-table).

In [1]:
# Run from the folder where you ran `aatm init` (it holds .aatm/ and .env).
import os

if not os.path.isdir(".aatm") and os.path.isdir("../.aatm"):
    os.chdir("..")

## 1. Mapper and notes

The same mapper as in notebook 04 codes the extracted names.

In [2]:
from aatm.omop import ClinicalNote, OmopExtractionPipeline
from aatm.selectors import OpenAILLMSelector
from aatm.terminology_mapper import TerminologyMapper

mapper = TerminologyMapper(
    retriever="text-embedding-3-small", selector=OpenAILLMSelector("gpt-5-mini")
)
mapper.retriever.top_k = 30

notes = [
    ClinicalNote(
        text=(
            "Home meds: lisinopril 10 mg PO daily since 2020, metformin 500 mg tablet BID.\n"
            "Ceftriaxone 1 g IV given in the ED.\n"
            "Plan: start amoxicillin 875 mg PO BID for 7 days. Consider a statin.\n"
            "Allergic to penicillin. Not on anticoagulation."
        ),
        person_id=30,
        note_date="2025-06-01",
        visit_occurrence_id=900,
        provider_id=12,
    )
]

## 2. Run

In [3]:
pipeline = OmopExtractionPipeline(
    "drug_exposure", model_id="openai/gpt-5-mini", mapper=mapper
)
tables = await pipeline(notes)

tables["drug_exposure"][
    [
        "drug_source_value",
        "drug_concept_id",
        "drug_exposure_start_date",
        "drug_exposure_end_date",
        "route_concept_id",
        "sig",
        "visit_occurrence_id",
    ]
]

,drug_source_value,drug_concept_id,drug_exposure_start_date,drug_exposure_end_date,route_concept_id,sig,visit_occurrence_id
0,lisinopril 10 mg,1308304,2020-01-01,2025-06-01,4132161,10 mg PO daily,<NA>
1,metformin 500 mg tablet,40164929,2025-06-01,2025-06-01,<NA>,500 mg tablet BID,<NA>
2,Ceftriaxone 1 g,46287328,2025-06-01,2025-06-01,4171047,NaN,900
3,amoxicillin 875 mg,19083494,2025-06-01,2025-06-07,4132161,875 mg PO BID for 7 days,900


"Consider a statin" is only considered, penicillin is an allergy and anticoagulation is
negated, so none of them is a row.

**End dates** are required by the CDM. When the note gives none they are inferred:
start + duration - 1 ("for 7 days"), else start + 29 days for `prescribed`, the note date
for `taking`, and the start date for `administered`.

**Route** (`oral`, `intravenous`, ...) is stored as its standard SNOMED route concept.

## 3. Check the coded concepts and the issues

The CDM prefers the most detailed concept the text supports (a clinical drug with strength and form over an ingredient). Review low scores.

In [4]:
tables.mappings_dataframe()[
    ["row_id", "column", "text", "concept_id", "concept_name", "domain_id", "score"]
]

,row_id,column,text,concept_id,concept_name,domain_id,score
0,1,drug_concept_id,lisinopril 10 mg,1308304,lisinopril 10 MG,Drug,0.900960
1,2,drug_concept_id,metformin 500 mg tablet,40164929,metformin hydrochloride 500 MG Oral Tablet,Drug,0.803700
2,3,drug_concept_id,Ceftriaxone 1 g,46287328,ceftriaxone 1000 MG Injection,Drug,0.587347
3,4,drug_concept_id,amoxicillin 875 mg,19083494,amoxicillin 875 MG,Drug,0.938107


In [5]:
tables.issues_dataframe()[["row_id", "column", "reason", "detail"]]

,row_id,column,reason,detail
0,1,drug_exposure_start_date,imputed_date_parts,'2020' has no day or month; read as 2020-01-01


## Next

Implants, equipment and supplies: [08_device_exposure.ipynb](08_device_exposure.ipynb).